In [224]:
import pandas as pd
import numpy as np
import zipfile as zipfile
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#from surprise.model_selection import train_test_split
#import matplotlib.pyplot as plt
#%matplotlib inline
#import sys
#import pickle
#import requests
#from numpy import genfromtxt

## Preprocess

### Mapping movies

In [225]:
# Common films between MDL and IMDB datasets
imdb_mdl_mapp = pd.read_csv('data_processed/imdb_mdl_mapp.csv', 
                            dtype={'tconst':object, 'movieId':object, 'primaryTitle':object, 'startYear':int})


# Reference list to limit datasets
imdb_mdl_mapp = imdb_mdl_mapp[(imdb_mdl_mapp['startYear']>2000) & (imdb_mdl_mapp['startYear']<2010)]
list_movies_imdb = imdb_mdl_mapp['tconst'].tolist()

# Drop unused columns
imdb_mdl_mapp = imdb_mdl_mapp.drop(['Unnamed: 0','primaryTitle', 'startYear'], axis=1)

print(imdb_mdl_mapp.info())
imdb_mdl_mapp.head(5)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 8501 entries, 1638 to 41309
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   tconst   8501 non-null   object
 1   movieId  8501 non-null   object
dtypes: object(2)
memory usage: 199.2+ KB
None


,tconst,movieId
1638,tt0035423,4992
9296,tt0096056,116293
12324,tt0118141,84794
12360,tt0118589,4775
12455,tt0118926,6335


### Movie Data Lens content

In [226]:
# Load movie dataset
mdl_content = pd.read_csv('data_processed/mdl_content.csv.zip',
                          compression = 'zip', 
                          sep = ',',
                          usecols = ['movieId', 'title', 'genres'])

mdl_content['movieId'] = mdl_content['movieId'].astype(object)


print(mdl_content.info())
mdl_content.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8109 entries, 0 to 8108
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  8109 non-null   object
 1   title    8109 non-null   object
 2   genres   8109 non-null   object
dtypes: object(3)
memory usage: 190.2+ KB
None


,movieId,title,genres
0,24,Powder,Drama Sci-Fi
1,32,Twelve Monkeys (a.k.a. 12 Monkeys),Mystery Sci-Fi Thriller
2,43,Restoration,Drama
3,51,Guardian Angel,Action Drama Thriller
4,73,"Misérables, Les",Drama War


### IMDB content

In [227]:
# Load /  A ajouter : chargement direct dpuis url
imdb_content = pd.read_csv('data_processed/imdb_content.csv.zip',
                           compression='zip', 
                           sep=',')

imdb_content = imdb_content.drop(['genres'], axis=1)

print(imdb_content.info())
imdb_content.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8501 entries, 0 to 8500
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   tconst           8501 non-null   object
 1   titleType        8501 non-null   object
 2   primaryTitle     8501 non-null   object
 3   startYear        8501 non-null   int64 
 4   runtimeMinutes   8501 non-null   int64 
 5   runtimeCategory  8501 non-null   object
 6   yearCategory     8501 non-null   object
 7   directorName     6989 non-null   object
dtypes: int64(2), object(6)
memory usage: 531.4+ KB
None


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,runtimeCategory,yearCategory,directorName
0,tt0035423,movie,Kate & Leopold,2001,118,F,I,James Mangold
1,tt0096056,movie,Crime and Punishment,2002,126,G,I,Menahem Golan
2,tt0118141,movie,What Is It?,2005,72,F,I,Crispin Glover
3,tt0118589,movie,Glitter,2001,104,F,I,Vondie Curtis-Hall
4,tt0118926,movie,The Dancer Upstairs,2002,132,G,I,John Malkovich


### Merge

In [228]:
# Merge of MDL and IMDB data
df = imdb_mdl_mapp.merge(right = imdb_content, left_on='tconst', right_on='tconst', how='left').reset_index()
df = df.merge(right = mdl_content, left_on='movieId', right_on='movieId', how='left').reset_index()

df = df.drop(['index', 'level_0'], axis=1)

# Rearrange column order
#df = df[['movieId', 'tconst', 'title', 'year', 'genres', 'directorName']]

print(df.info())
df.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8501 entries, 0 to 8500
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   tconst           8501 non-null   object
 1   movieId          8501 non-null   object
 2   titleType        8501 non-null   object
 3   primaryTitle     8501 non-null   object
 4   startYear        8501 non-null   int64 
 5   runtimeMinutes   8501 non-null   int64 
 6   runtimeCategory  8501 non-null   object
 7   yearCategory     8501 non-null   object
 8   directorName     6989 non-null   object
 9   title            0 non-null      object
 10  genres           0 non-null      object
dtypes: int64(2), object(9)
memory usage: 730.7+ KB
None


,tconst,movieId,titleType,primaryTitle,startYear,runtimeMinutes,runtimeCategory,yearCategory,directorName,title,genres
0,tt0035423,4992,movie,Kate & Leopold,2001,118,F,I,James Mangold,NaN,NaN
1,tt0096056,116293,movie,Crime and Punishment,2002,126,G,I,Menahem Golan,NaN,NaN
2,tt0118141,84794,movie,What Is It?,2005,72,F,I,Crispin Glover,NaN,NaN
3,tt0118589,4775,movie,Glitter,2001,104,F,I,Vondie Curtis-Hall,NaN,NaN
4,tt0118926,6335,movie,The Dancer Upstairs,2002,132,G,I,John Malkovich,NaN,NaN


In [229]:
df.duplicated().sum()

0

In [230]:
df.isna().sum()

tconst                0
movieId               0
titleType             0
primaryTitle          0
startYear             0
runtimeMinutes        0
runtimeCategory       0
yearCategory          0
directorName       1512
title              8501
genres             8501
dtype: int64

### Features build

In [232]:
#df['year'] = df['year'].astype('str')

df['directorName'] = df['directorName'].astype('str')

df['combined_features'] = df[['titleType', 'runtimeCategory', 'yearCategory', 'directorName']].apply(lambda x: ' '.join(x), axis=1)


print(df.info())
df.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8501 entries, 0 to 8500
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   tconst             8501 non-null   object
 1   movieId            8501 non-null   object
 2   titleType          8501 non-null   object
 3   primaryTitle       8501 non-null   object
 4   startYear          8501 non-null   int64 
 5   runtimeMinutes     8501 non-null   int64 
 6   runtimeCategory    8501 non-null   object
 7   yearCategory       8501 non-null   object
 8   directorName       8501 non-null   object
 9   title              0 non-null      object
 10  genres             0 non-null      object
 11  combined_features  8501 non-null   object
dtypes: int64(2), object(10)
memory usage: 797.1+ KB
None


,tconst,movieId,titleType,primaryTitle,startYear,runtimeMinutes,runtimeCategory,yearCategory,directorName,title,genres,combined_features
0,tt0035423,4992,movie,Kate & Leopold,2001,118,F,I,James Mangold,NaN,NaN,movie F I James Mangold
1,tt0096056,116293,movie,Crime and Punishment,2002,126,G,I,Menahem Golan,NaN,NaN,movie G I Menahem Golan
2,tt0118141,84794,movie,What Is It?,2005,72,F,I,Crispin Glover,NaN,NaN,movie F I Crispin Glover
3,tt0118589,4775,movie,Glitter,2001,104,F,I,Vondie Curtis-Hall,NaN,NaN,movie F I Vondie Curtis-Hall
4,tt0118926,6335,movie,The Dancer Upstairs,2002,132,G,I,John Malkovich,NaN,NaN,movie G I John Malkovich


### Tokenization / Features extraction

In [233]:
cv = CountVectorizer()
count_matrix = cv.fit_transform(df["combined_features"])
print("Count Matrix:", count_matrix.toarray())

Count Matrix: [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


## Model

In [234]:
cosine_sim = cosine_similarity(count_matrix)

In [235]:
print("Min value : ",cosine_sim.min(),"\nMax value : " ,cosine_sim.max(),"\nAverage value : " ,cosine_sim.mean(),"\nMedian value : " ,np.median(cosine_sim))

cosine_sim[:9,:9]

Min value :  0.0 
Max value :  1.0000000000000002 
Average value :  0.21363714171196846 
Median value :  0.3333333333333334


array([[1.        , 0.33333333, 0.33333333, 0.28867513, 0.33333333,
        0.40824829, 0.33333333, 0.33333333, 0.40824829],
       [0.33333333, 1.        , 0.33333333, 0.28867513, 0.33333333,
        0.40824829, 0.33333333, 0.33333333, 0.40824829],
       [0.33333333, 0.33333333, 1.        , 0.28867513, 0.33333333,
        0.40824829, 0.33333333, 0.33333333, 0.40824829],
       [0.28867513, 0.28867513, 0.28867513, 1.        , 0.28867513,
        0.35355339, 0.28867513, 0.28867513, 0.35355339],
       [0.33333333, 0.33333333, 0.33333333, 0.28867513, 1.        ,
        0.40824829, 0.33333333, 0.33333333, 0.40824829],
       [0.40824829, 0.40824829, 0.40824829, 0.35355339, 0.40824829,
        1.        , 0.40824829, 0.40824829, 0.5       ],
       [0.33333333, 0.33333333, 0.33333333, 0.28867513, 0.33333333,
        0.40824829, 1.        , 0.33333333, 0.40824829],
       [0.33333333, 0.33333333, 0.33333333, 0.28867513, 0.33333333,
        0.40824829, 0.33333333, 1.        , 0.40824829],


### Save of the model

In [236]:
# Save of the model

## Test

In [237]:
movie_user_likes = "Crime and Punishment"

def get_index_from_title(title):
    return df[df.primaryTitle == title].index.values[0]
    
movie_index = get_index_from_title(movie_user_likes)

movie_index

1

In [238]:
similar_movies = list(enumerate(cosine_sim[movie_index]))

sorted_similar_movies = sorted(similar_movies, key=lambda x:x[1], reverse=True)[:10]

sorted_similar_movies

[(1, 1.0000000000000002),
 (5, 0.408248290463863),
 (8, 0.408248290463863),
 (13, 0.408248290463863),
 (17, 0.408248290463863),
 (21, 0.408248290463863),
 (22, 0.408248290463863),
 (46, 0.408248290463863),
 (48, 0.408248290463863),
 (52, 0.408248290463863)]

In [255]:
def get_title_from_index(index):
    return str(df.iloc[index][3])

def get_year_from_index(index):
    return str(df.iloc[index][4])

def get_genre_from_index(index):
    return str(df.iloc[index][10])

print("index 0: ",get_title_from_index(0)) 



index 0:  Kate & Leopold


In [264]:
for movie in sorted_similar_movies:
    #print(get_title_from_index(movie[0]))
    print(get_title_from_index(movie[0]) + '(' + get_year_from_index(movie[0]))

Crime and Punishment(2002
Don's Plum(2001
From Hell(2001
Corpse Bride(2005
Shrek(2001
Treasure Planet(2002
Intolerable Cruelty(2003
Vidocq(2001
Sinbad: Legend of the Seven Seas(2003
Spirit: Stallion of the Cimarron(2002
